In [2]:
import os 
import sys
import time
# Set environment variables for PySpark to run correctly on Windows
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
os.environ['SPARK_SUBMIT_OPTS'] = '-Djava.security.manager=allow'

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc

In [3]:
start_time = time.time()

spark = SparkSession.builder \
    .appName("Yelp_User_Analysis") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.driver.extraJavaOptions", "-Djava.security.manager=allow") \
    .config("spark.executor.extraJavaOptions", "-Djava.security.manager=allow") \
    .config("spark.sql.shuffle.partitions", "10") \
    .getOrCreate()

print(f"Spark Session initialized in {time.time() - start_time:.2f} seconds.")
print(f"Spark Version: {spark.version}")

Spark Session initialized in 14.12 seconds.
Spark Version: 4.1.2


## 1. Dataset ``data\raw\yelp_academic_dataset_review.json``

In [4]:
review_df = spark.read.json("data/raw/yelp_academic_dataset_review.json")
review_df.show(5, truncate=False)

+----------------------+----+-------------------+-----+----------------------+-----+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------+----------------------+
|business_id           |cool|date               |fu

In [5]:
review_df.printSchema()

root
 |-- business_id: string (nullable = true)
 |-- cool: long (nullable = true)
 |-- date: string (nullable = true)
 |-- funny: long (nullable = true)
 |-- review_id: string (nullable = true)
 |-- stars: double (nullable = true)
 |-- text: string (nullable = true)
 |-- useful: long (nullable = true)
 |-- user_id: string (nullable = true)



In [8]:
print("Calculating total user count...")
start_time = time.time()
total_users = review_df.select("review_id").distinct().count()
print(f"Total Users in Dataset: {total_users:,} (calculated in {time.time() - start_time:.2f} seconds)")

Calculating total user count...
Total Users in Dataset: 6,990,280 (calculated in 8.71 seconds)


## 2. Dataset ``data\raw\yelp_academic_dataset_checkin.json``

In [9]:
checkin_df = spark.read.json("data/raw/yelp_academic_dataset_checkin.json")
checkin_df.show(5, truncate=False)

+----------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|business_id           |date                                                                                                                                                                                                                                                                                                                                                                                                                  

In [10]:
checkin_df.printSchema()

root
 |-- business_id: string (nullable = true)
 |-- date: string (nullable = true)



In [13]:
print("Calculating total user count...")
start_time = time.time()
total_users = checkin_df.select("business_id").distinct().count()
print(f"Total Businesses in Dataset: {total_users:,} (calculated in {time.time() - start_time:.2f} seconds)")

Calculating total user count...
Total Businesses in Dataset: 131,930 (calculated in 0.64 seconds)


## 3. Dataset ``data\raw\yelp_academic_dataset_business.json``

In [14]:
business_df = spark.read.json("data/raw/yelp_academic_dataset_business.json")
business_df.show(5, truncate=False)

+-------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------+----------------------------------------------------------------------------------------------------------+-------------+----------------------------------------------------------------------+-------+----------+------------+------------------------+-----------+------------+-----+-----+
|address                        |attributes                                                                                                                                                                                                                                                                                                    

In [15]:
business_df.printSchema()

root
 |-- address: string (nullable = true)
 |-- attributes: struct (nullable = true)
 |    |-- AcceptsInsurance: string (nullable = true)
 |    |-- AgesAllowed: string (nullable = true)
 |    |-- Alcohol: string (nullable = true)
 |    |-- Ambience: string (nullable = true)
 |    |-- BYOB: string (nullable = true)
 |    |-- BYOBCorkage: string (nullable = true)
 |    |-- BestNights: string (nullable = true)
 |    |-- BikeParking: string (nullable = true)
 |    |-- BusinessAcceptsBitcoin: string (nullable = true)
 |    |-- BusinessAcceptsCreditCards: string (nullable = true)
 |    |-- BusinessParking: string (nullable = true)
 |    |-- ByAppointmentOnly: string (nullable = true)
 |    |-- Caters: string (nullable = true)
 |    |-- CoatCheck: string (nullable = true)
 |    |-- Corkage: string (nullable = true)
 |    |-- DietaryRestrictions: string (nullable = true)
 |    |-- DogsAllowed: string (nullable = true)
 |    |-- DriveThru: string (nullable = true)
 |    |-- GoodForDancing: str